In [1]:
import pandas as pd
import os
import mysql.connector

# =========================================================
# 1) LOAD CSV FILES
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

dfs = []

for f in files:
    df = pd.read_csv(os.path.join(folder_path, f))
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Data loaded ✔")


# =========================================================
# 2) CLEAN DATA
# =========================================================

data.columns = data.columns.str.lower().str.strip()
data = data.dropna()

print("Data cleaned ✔")


# =========================================================
# 3) CREATE IDS
# =========================================================

data["customer_id"] = pd.factorize(data["currencytype"])[0] + 1
data["product_id"] = pd.factorize(data["productname"])[0] + 1
data["store_id"] = pd.factorize(data.index)[0] + 1


# =========================================================
# 4) SPLIT TABLES
# =========================================================

customers = data[["customer_id"]].drop_duplicates()

products = data[["product_id", "productname", "unit_price"]].drop_duplicates()

stores = data[["store_id"]].drop_duplicates()

sales = data[[
    "date",
    "qty",
    "customer_id",
    "product_id",
    "store_id",
    "currencytype"
]]


print("Tables split ✔")


# =========================================================
# 5) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales_db")

print("Connected ✔")


# =========================================================
# 6) CREATE TABLES
# =========================================================

cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(255),
    unit_price FLOAT
)
""")

cursor.execute("DROP TABLE IF EXISTS stores")
cursor.execute("""
CREATE TABLE stores (
    store_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS sales")
cursor.execute("""
CREATE TABLE sales (
    id INT AUTO_INCREMENT PRIMARY KEY,
    date VARCHAR(50),
    quantity FLOAT,
    customer_id INT,
    product_id INT,
    store_id INT,
    currency_type VARCHAR(50)
)
""")

conn.commit()

print("Tables created ✔")


# =========================================================
# 7) INSERT FUNCTION
# =========================================================

def insert(df, table):
    cols = ",".join(df.columns)
    values = ",".join(["%s"] * len(df.columns))

    query = f"INSERT INTO {table} ({cols}) VALUES ({values})"

    for row in df.itertuples(index=False):
        cursor.execute(query, tuple(row))

    conn.commit()


# =========================================================
# 8) INSERT DATA
# =========================================================

insert(customers, "customers")
insert(products, "products")
insert(stores, "stores")
insert(sales, "sales")

print("Data inserted ✔")


# =========================================================
# 9) CHECK
# =========================================================

cursor.execute("SHOW TABLES")
print(cursor.fetchall())


# =========================================================
# 10) CLOSE
# =========================================================

cursor.close()
conn.close()

print("DONE 🚀")

Data loaded ✔
Data cleaned ✔


KeyError: "['date'] not in index"

In [2]:
print(data.columns)

Index(['productname', 'qty', 'unit_price', 'saledate', 'currencytype',
       'customerid', 'storeid', 'customer_id', 'product_id', 'store_id'],
      dtype='str')


In [3]:
import pandas as pd
import os
import mysql.connector

# =========================================================
# 1) LOAD CSV FILES
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

dfs = []

for f in files:
    df = pd.read_csv(os.path.join(folder_path, f))
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Data loaded ✔")


# =========================================================
# 2) CLEAN COLUMN NAMES
# =========================================================

data.columns = (
    data.columns
    .str.lower()
    .str.strip()
)

print("Columns:", data.columns)


# =========================================================
# 3) CREATE IDS (IF NOT EXIST ALREADY)
# =========================================================

# use existing columns if already created
data["customer_id"] = data["customerid"]
data["product_id"] = data["product_id"]
data["store_id"] = data["store_id"]


# =========================================================
# 4) SPLIT TABLES
# =========================================================

customers = data[["customer_id"]].drop_duplicates()

products = data[["product_id", "productname", "unit_price"]].drop_duplicates()

stores = data[["store_id"]].drop_duplicates()

sales = data[[
    "saledate",
    "qty",
    "customer_id",
    "product_id",
    "store_id",
    "currencytype"
]]

print("Tables split ✔")


# =========================================================
# 5) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales_db")

print("Connected ✔")


# =========================================================
# 6) CREATE TABLES
# =========================================================

cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    productname VARCHAR(255),
    unit_price FLOAT
)
""")

cursor.execute("DROP TABLE IF EXISTS stores")
cursor.execute("""
CREATE TABLE stores (
    store_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS sales")
cursor.execute("""
CREATE TABLE sales (
    id INT AUTO_INCREMENT PRIMARY KEY,
    saledate VARCHAR(50),
    qty FLOAT,
    customer_id INT,
    product_id INT,
    store_id INT,
    currencytype VARCHAR(50)
)
""")

conn.commit()

print("Tables created ✔")


# =========================================================
# 7) INSERT FUNCTION
# =========================================================

def insert(df, table):
    cols = ",".join(df.columns)
    placeholders = ",".join(["%s"] * len(df.columns))

    query = f"INSERT INTO {table} ({cols}) VALUES ({placeholders})"

    for row in df.itertuples(index=False):
        cursor.execute(query, tuple(row))

    conn.commit()


# =========================================================
# 8) INSERT DATA
# =========================================================

insert(customers, "customers")
insert(products, "products")
insert(stores, "stores")
insert(sales, "sales")

print("Data inserted ✔")


# =========================================================
# 9) VERIFY
# =========================================================

cursor.execute("SHOW TABLES")
print(cursor.fetchall())


# =========================================================
# 10) CLOSE CONNECTION
# =========================================================

cursor.close()
conn.close()

print("DONE 🚀")

Data loaded ✔
Columns: Index(['productname', 'qty', 'unit_price', 'saledate', 'currencytype',
       'customerid', 'storeid'],
      dtype='str')


KeyError: 'product_id'

In [ ]:
import pandas as pd
import os
import mysql.connector

# =========================================================
# 1) LOAD DATA
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

dfs = []

for f in files:
    df = pd.read_csv(os.path.join(folder_path, f))
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Data loaded ✔")


# =========================================================
# 2) CLEAN COLUMN NAMES
# =========================================================

data.columns = data.columns.str.lower().str.strip()

data["unit_price"] = (
    data["unit_price"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
)

data["unit_price"] = pd.to_numeric(data["unit_price"], errors="coerce")

print("Columns:", data.columns)


# =========================================================
# 3) CREATE IDS (IMPORTANT FIX)
# =========================================================

data["customer_id"] = pd.factorize(data["customerid"])[0] + 1
data["product_id"] = pd.factorize(data["productname"])[0] + 1
data["store_id"] = pd.factorize(data["storeid"])[0] + 1


# =========================================================
# 4) SPLIT TABLES
# =========================================================

customers = data[["customer_id"]].drop_duplicates()

products = data[["product_id", "productname", "unit_price"]].drop_duplicates()

stores = data[["store_id"]].drop_duplicates()

sales = data[[
    "saledate",
    "qty",
    "customer_id",
    "product_id",
    "store_id",
    "currencytype"
]]

print("Tables split ✔")


# =========================================================
# 5) MYSQL CONNECTION
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales1_db")

print("Connected ✔")


# =========================================================
# 6) CREATE TABLES
# =========================================================

cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    productname VARCHAR(255),
    unit_price FLOAT
)
""")

cursor.execute("DROP TABLE IF EXISTS stores")
cursor.execute("""
CREATE TABLE stores (
    store_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS sales")
cursor.execute("""
CREATE TABLE sales (
    id INT AUTO_INCREMENT PRIMARY KEY,
    saledate VARCHAR(50),
    qty FLOAT,
    customer_id INT,
    product_id INT,
    store_id INT,
    currencytype VARCHAR(50)
)
""")

conn.commit()

print("Tables created ✔")


# =========================================================
# 7) INSERT FUNCTION
# =========================================================
def insert(df, table):

    df = df.copy()

    df = df.fillna("Unknown")

    cols = ",".join(df.columns)
    placeholders = ",".join(["%s"] * len(df.columns))

    query = f"INSERT INTO {table} ({cols}) VALUES ({placeholders})"

    for row in df.itertuples(index=False):

        values = tuple(
            None if pd.isna(x) else x
            for x in row
        )

        cursor.execute(query, values)

    conn.commit()

# =========================================================
# 8) INSERT DATA
# =========================================================

insert(customers, "customers")
insert(products, "products")
insert(stores, "stores")
insert(sales, "sales")

print("Data inserted ✔")


# =========================================================
# 9) VERIFY
# =========================================================

cursor.execute("SHOW TABLES")
print(cursor.fetchall())


# =========================================================
# 10) CLOSE
# =========================================================

cursor.close()
conn.close()

print("DONE 🚀")

Data loaded ✔
Columns: Index(['productname', 'qty', 'unit_price', 'saledate', 'currencytype',
       'customerid', 'storeid'],
      dtype='str')
Tables split ✔
Connected ✔


In [4]:
import pandas as pd
import os
import mysql.connector

# =========================================================
# 1) EXTRACT (LOAD CSV FILES)
# =========================================================

folder_path = r"C:\Users\DELL\wajd-mq\sales"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

dfs = []

for f in files:
    df = pd.read_csv(os.path.join(folder_path, f))
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Data loaded ✔")


# =========================================================
# 2) CLEAN COLUMN NAMES
# =========================================================

data.columns = data.columns.str.lower().str.strip()

print("Columns:", data.columns)


# =========================================================
# 3) TRANSFORM (DATA CLEANING)
# =========================================================

# ---- numeric conversion
data["qty"] = pd.to_numeric(data["qty"], errors="coerce")
data["unit_price"] = pd.to_numeric(data["unit_price"], errors="coerce")

# ---- datetime conversion
data["saledate"] = pd.to_datetime(data["saledate"], errors="coerce")

# ---- fill missing values
data["qty"] = data["qty"].fillna(0)
data["unit_price"] = data["unit_price"].fillna(data["unit_price"].median())

# ---- text cleaning
data["productname"] = data["productname"].str.strip().str.lower()
data["currencytype"] = data["currencytype"].str.strip().str.upper()

# ---- feature engineering
data["total_price"] = data["qty"] * data["unit_price"]

# ---- currency conversion (OMR example)
rate_usd_to_omr = 0.385
data["total_price_omr"] = data["total_price"] * rate_usd_to_omr


# =========================================================
# 4) CREATE IDS
# =========================================================

data["customer_id"] = pd.factorize(data["customerid"])[0] + 1
data["product_id"] = pd.factorize(data["productname"])[0] + 1
data["store_id"] = pd.factorize(data["storeid"])[0] + 1


# =========================================================
# 5) SPLIT INTO TABLES
# =========================================================

customers = (
    data[["customer_id"]]
    .drop_duplicates(subset=["customer_id"])
)

products = (
    data[["product_id", "productname", "unit_price"]]
    .drop_duplicates(subset=["product_id"])
)

stores = (
    data[["store_id"]]
    .drop_duplicates(subset=["store_id"])
)

sales = data[[
    "saledate",
    "qty",
    "customer_id",
    "product_id",
    "store_id",
    "currencytype",
    "total_price",
    "total_price_omr"
]]

# =========================================================
# 6) CONNECT TO MYSQL
# =========================================================

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS sales_db")
cursor.execute("USE sales1_db")

print("Connected ✔")


# =========================================================
# 7) CREATE TABLES
# =========================================================

cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    productname VARCHAR(255),
    unit_price FLOAT
)
""")

cursor.execute("DROP TABLE IF EXISTS stores")
cursor.execute("""
CREATE TABLE stores (
    store_id INT PRIMARY KEY
)
""")

cursor.execute("DROP TABLE IF EXISTS sales")
cursor.execute("""
CREATE TABLE sales (
    id INT AUTO_INCREMENT PRIMARY KEY,
    saledate DATETIME,
    qty FLOAT,
    customer_id INT,
    product_id INT,
    store_id INT,
    currencytype VARCHAR(50),
    total_price FLOAT,
    total_price_omr FLOAT
)
""")

conn.commit()

print("Tables created ✔")


# =========================================================
# 8) INSERT FUNCTION (SAFE)
# =========================================================

def insert(df, table):

    df = df.copy()

    # convert NaN -> None (SQL NULL)
    df = df.where(pd.notnull(df), None)

    cols = ",".join(df.columns)
    placeholders = ",".join(["%s"] * len(df.columns))

    query = f"INSERT INTO {table} ({cols}) VALUES ({placeholders})"

    for row in df.itertuples(index=False):
        cursor.execute(query, tuple(row))

    conn.commit()


# =========================================================
# 9) LOAD DATA INTO MYSQL
# =========================================================

insert(customers, "customers")
insert(products, "products")
insert(stores, "stores")
insert(sales, "sales")

print("Data inserted ✔")


# =========================================================
# 10) VERIFY
# =========================================================

cursor.execute("SHOW TABLES")
print(cursor.fetchall())


# =========================================================
# 11) CLOSE CONNECTION
# =========================================================

cursor.close()
conn.close()

print("DONE 🚀")

Data loaded ✔
Columns: Index(['productname', 'qty', 'unit_price', 'saledate', 'currencytype',
       'customerid', 'storeid'],
      dtype='str')
Connected ✔
Tables created ✔


ProgrammingError: 1054 (42S22): Unknown column 'nan' in 'field list'

In [1]:
#=========================================================
# COMPLETE MULTI-STORE ETL PIPELINE
# =========================================================
# FIXED VERSION
#
# PROBLEM SOLVED:
# Duplicate StoreID values such as:
# STORE_A
# store-A
# Store_A
#
# SOLUTION:
# Generate UNIQUE IDs for:
# - Product
# - Customer
# - Store
# - Sale
#
# Even if CSV files do not contain IDs
# =========================================================

# =========================================================
# INSTALL LIBRARIES
# =========================================================
# pip install pandas sqlalchemy pymysql scikit-learn

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import glob
import os
import uuid

from sqlalchemy import create_engine, text

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# =========================================================
# DATABASE CONNECTION
# =========================================================

username = "root"
password = "1234"
host = "localhost"
database = "salesdb"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

# =========================================================
# EXTRACT FUNCTION
# =========================================================

def extract_data(folder_path):

    print("\n==============================")
    print("EXTRACTING CSV FILES")
    print("==============================")

    csv_files = glob.glob(
        os.path.join(folder_path, "*.csv")
    )

    all_dataframes = []

    for file in csv_files:

        print(f"Reading File: {file}")

        df = pd.read_csv(file)

        # Add source filename
        df["Source_File"] = os.path.basename(file)

        all_dataframes.append(df)

    combined_data = pd.concat(
        all_dataframes,
        ignore_index=True
    )

    print("\nFILES MERGED SUCCESSFULLY")

    print(combined_data.head())

    return combined_data

# =========================================================
# TRANSFORM FUNCTION
# =========================================================

def transform_data(data):

    print("\n==============================")
    print("TRANSFORMING DATA")
    print("==============================")

    # -----------------------------------------------------
    # REMOVE DUPLICATES
    # -----------------------------------------------------

    data = data.drop_duplicates()

    # -----------------------------------------------------
    # HANDLE MISSING VALUES
    # -----------------------------------------------------

    print("\nHandling Missing Values...")

    data["Qty"] = data["Qty"].fillna(
        data["Qty"].mean()
    )

    data["Unit_Price"] = data["Unit_Price"].fillna(
        data["Unit_Price"].mean()
    )

    data = data.dropna(
        subset=["CustomerID"]
    )

    # -----------------------------------------------------
    # DATA TYPE CONVERSION
    # -----------------------------------------------------

    print("Converting Data Types...")

    data["Qty"] = data["Qty"].astype(int)

    data["Unit_Price"] = data["Unit_Price"].astype(float)

    data["CustomerID"] = data["CustomerID"].astype(str)

    data["StoreID"] = data["StoreID"].astype(str)

    data["SaleDate"] = pd.to_datetime(
        data["SaleDate"]
    )

    # -----------------------------------------------------
    # TEXT CLEANING
    # -----------------------------------------------------

    print("Cleaning Text Fields...")

    text_columns = [

        "ProductName",
        "CurrencyType",
        "CustomerID",
        "StoreID"

    ]

    for col in text_columns:

        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    # -----------------------------------------------------
    # GENERATE UNIQUE PRODUCT IDS
    # -----------------------------------------------------

    print("Generating Product IDs...")

    unique_products = data["ProductName"].unique()

    product_mapping = {}

    for product in unique_products:

        product_mapping[product] = str(uuid.uuid4())

    data["ProductID"] = data["ProductName"].map(
        product_mapping
    )

    # -----------------------------------------------------
    # GENERATE UNIQUE STORE IDS
    # -----------------------------------------------------

    print("Generating Store IDs...")

    unique_stores = data["StoreID"].unique()

    store_mapping = {}

    for store in unique_stores:

        store_mapping[store] = str(uuid.uuid4())

    data["GeneratedStoreID"] = data["StoreID"].map(
        store_mapping
    )

    # -----------------------------------------------------
    # GENERATE UNIQUE CUSTOMER IDS
    # -----------------------------------------------------

    print("Generating Customer IDs...")

    unique_customers = data["CustomerID"].unique()

    customer_mapping = {}

    for customer in unique_customers:

        customer_mapping[customer] = str(uuid.uuid4())

    data["GeneratedCustomerID"] = data["CustomerID"].map(
        customer_mapping
    )

    # -----------------------------------------------------
    # GENERATE SALE IDs
    # -----------------------------------------------------

    print("Generating Sale IDs...")

    data["SaleID"] = [

        str(uuid.uuid4())
        for _ in range(len(data))

    ]

    # -----------------------------------------------------
    # CURRENCY CONVERSION
    # -----------------------------------------------------

    print("Converting Currency to OMR...")

    exchange_rates = {

        "usd": 0.385,
        "eur": 0.420,
        "omr": 1

    }

    data["Unit_Price_OMR"] = data.apply(

        lambda row:
        row["Unit_Price"] *
        exchange_rates.get(
            row["CurrencyType"],
            1
        ),

        axis=1
    )

    # -----------------------------------------------------
    # CREATE TOTAL PRICE
    # -----------------------------------------------------

    print("Creating Total_Price...")

    data["Total_Price"] = (

        data["Qty"] *
        data["Unit_Price_OMR"]

    )

    # -----------------------------------------------------
    # SKLEARN PIPELINE
    # -----------------------------------------------------

    print("Applying sklearn Pipeline...")

    numeric_features = [

        "Qty",
        "Unit_Price_OMR",
        "Total_Price"

    ]

    numeric_pipeline = Pipeline([

        ("imputer", SimpleImputer(strategy="mean")),

        ("scaler", StandardScaler())

    ])

    preprocessor = ColumnTransformer([

        ("num", numeric_pipeline, numeric_features)

    ])

    transformed = preprocessor.fit_transform(data)

    transformed_df = pd.DataFrame(

        transformed,
        columns=numeric_features

    )

    data[numeric_features] = transformed_df

    print("\nTRANSFORMATION COMPLETED")

    print(data.head())

    return data

# =========================================================
# CREATE TABLES FUNCTION
# =========================================================

def create_tables():

    print("\n==============================")
    print("CREATING TABLES")
    print("==============================")

    with engine.connect() as conn:

        # -------------------------------------------------
        # PRODUCT TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Product (

            ProductID VARCHAR(255) PRIMARY KEY,

            ProductName VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # CUSTOMER TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Customer (

            CustomerPK VARCHAR(255) PRIMARY KEY,

            OriginalCustomerID VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # STORE TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Store (

            StorePK VARCHAR(255) PRIMARY KEY,

            OriginalStoreID VARCHAR(255)

        )

        """))

        # -------------------------------------------------
        # SALE TABLE
        # -------------------------------------------------

        conn.execute(text("""

        CREATE TABLE IF NOT EXISTS Sale (

            SaleID VARCHAR(255) PRIMARY KEY,

            ProductID VARCHAR(255),

            CustomerPK VARCHAR(255),

            StorePK VARCHAR(255),

            Qty FLOAT,

            Unit_Price_OMR FLOAT,

            Total_Price FLOAT,

            SaleDate DATE,

            CurrencyType VARCHAR(50),

            Source_File VARCHAR(255),

            FOREIGN KEY (ProductID)
            REFERENCES Product(ProductID),

            FOREIGN KEY (CustomerPK)
            REFERENCES Customer(CustomerPK),

            FOREIGN KEY (StorePK)
            REFERENCES Store(StorePK)

        )

        """))

        conn.commit()

    print("TABLES CREATED SUCCESSFULLY")

# =========================================================
# LOAD FUNCTION
# =========================================================

def load_data(data):

    print("\n==============================")
    print("LOADING DATA")
    print("==============================")

    # -----------------------------------------------------
    # PRODUCT TABLE
    # -----------------------------------------------------

    product_df = data[[

        "ProductID",
        "ProductName"

    ]].drop_duplicates()

    # -----------------------------------------------------
    # CUSTOMER TABLE
    # -----------------------------------------------------

    customer_df = data[[

        "GeneratedCustomerID",
        "CustomerID"

    ]].drop_duplicates()

    customer_df.columns = [

        "CustomerPK",
        "OriginalCustomerID"

    ]

    # -----------------------------------------------------
    # STORE TABLE
    # -----------------------------------------------------

    store_df = data[[

        "GeneratedStoreID",
        "StoreID"

    ]].drop_duplicates()

    store_df.columns = [

        "StorePK",
        "OriginalStoreID"

    ]

    # -----------------------------------------------------
    # SALE TABLE
    # -----------------------------------------------------

    sale_df = data[[

        "SaleID",

        "ProductID",

        "GeneratedCustomerID",

        "GeneratedStoreID",

        "Qty",

        "Unit_Price_OMR",

        "Total_Price",

        "SaleDate",

        "CurrencyType",

        "Source_File"

    ]]

    sale_df.columns = [

        "SaleID",

        "ProductID",

        "CustomerPK",

        "StorePK",

        "Qty",

        "Unit_Price_OMR",

        "Total_Price",

        "SaleDate",

        "CurrencyType",

        "Source_File"

    ]

    # -----------------------------------------------------
    # LOAD DATA
    # -----------------------------------------------------

    product_df.to_sql(
        "Product",
        con=engine,
        if_exists="append",
        index=False
    )

    customer_df.to_sql(
        "Customer",
        con=engine,
        if_exists="append",
        index=False
    )

    store_df.to_sql(
        "Store",
        con=engine,
        if_exists="append",
        index=False
    )

    sale_df.to_sql(
        "Sale",
        con=engine,
        if_exists="append",
        index=False
    )

    print("DATA LOADED SUCCESSFULLY")

# =========================================================
# VERIFY FUNCTION
# =========================================================

def verify_data():

    print("\n==============================")
    print("VERIFYING DATA")
    print("==============================")

    query = """

    SELECT *
    FROM Sale

    """

    result = pd.read_sql(query, engine)

    print(result.head())

# =========================================================
# MAIN ETL FUNCTION
# =========================================================

def run_etl_pipeline():

    # STEP 1
    data = extract_data(r"C:\Users\DELL\wajd-mq\sales")

    # STEP 2
    transformed_data = transform_data(data)

    # STEP 3
    create_tables()

    # STEP 4
    load_data(transformed_data)

    # STEP 5
    verify_data()

    print("\nETL PIPELINE COMPLETED SUCCESSFULLY")

# =========================================================
# RUN PIPELINE
# =========================================================

run_etl_pipeline()



EXTRACTING CSV FILES
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_1.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_2.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_3.csv

FILES MERGED SUCCESSFULLY
          ProductName  Qty  Unit_Price    SaleDate CurrencyType  \
0         Smith Paper  3.0        10.5   7/13/2024          OMR   
1      Johnson Screen  NaN         NaN   2/23/2025          Usd   
2  Roberts Ingredient  3.0        30.0  11/13/2024          USD   
3       White Monitor  NaN        10.5   4/16/2025          USD   
4  Rodriguez Keyboard  2.0        20.0    8/3/2024          usd   

                             CustomerID  StoreID        Source_File  
0  9ca482a2-0356-49c1-b5e3-88ae98d1cc2f  Store_A  store_sales_1.csv  
1  c0b9df4e-8f03-4bf0-a31b-0a7d7c2a8907  Store_A  store_sales_1.csv  
2  97dc18e3-2c12-4e26-9863-32514e82e822  Store_A  store_sales_1.csv  
3  e4d09733-d496-47b3-a4b5-04de84d8fd06  Store_A  store_sales_1.csv  
4  435ecb46-4545-4

C:\Users\DELL\AppData\Local\Temp\ipykernel_16036\244911780.py:539: UserWarning: The provided table name 'Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  product_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_16036\244911780.py:546: UserWarning: The provided table name 'Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_16036\244911780.py:553: UserWarning: The provided table name 'Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  store_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_16036\244911780.py:560: UserWarning: The provided table name 'Sale' is not found exactly as such in the 

In [1]:
# =========================================================
# COMPLETE MULTI-STORE ETL PIPELINE
# =========================================================

import pandas as pd
import glob
import os
import uuid
import time

from sqlalchemy import create_engine, text
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# =========================================================
# DATABASE CONNECTION
# =========================================================

username = "root"
password = "1234"
host = "localhost"
database = "salesdb"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

# =========================================================
# EXTRACT FUNCTION
# =========================================================

def extract_data(folder_path):

    print("\n==============================")
    print("EXTRACTING CSV FILES")
    print("==============================")

    csv_files = glob.glob(
        os.path.join(folder_path, "*.csv")
    )

    all_dataframes = []

    for file in csv_files:

        print(f"Reading File: {file}")

        df = pd.read_csv(file)

        df["Source_File"] = os.path.basename(file)

        all_dataframes.append(df)

    if len(all_dataframes) == 0:
        raise ValueError("No CSV files found!")

    combined_data = pd.concat(
        all_dataframes,
        ignore_index=True
    )

    print("\nFILES MERGED SUCCESSFULLY")

    return combined_data

# =========================================================
# TRANSFORM FUNCTION
# =========================================================

def transform_data(data):

    print("\n==============================")
    print("TRANSFORMING DATA")
    print("==============================")

    data = data.drop_duplicates()

    data["Qty"] = data["Qty"].fillna(
        data["Qty"].mean()
    )

    data["Unit_Price"] = data["Unit_Price"].fillna(
        data["Unit_Price"].mean()
    )

    data = data.dropna(
        subset=["CustomerID"]
    )

    data["Qty"] = data["Qty"].astype(int)

    data["Unit_Price"] = data["Unit_Price"].astype(float)

    data["SaleDate"] = pd.to_datetime(
        data["SaleDate"]
    )

    for col in [
        "ProductName",
        "CurrencyType",
        "CustomerID",
        "StoreID"
    ]:

        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    data["ProductID"] = data["ProductName"].map(
        {
            p: str(uuid.uuid4())
            for p in data["ProductName"].unique()
        }
    )

    data["GeneratedCustomerID"] = data["CustomerID"].map(
        {
            c: str(uuid.uuid4())
            for c in data["CustomerID"].unique()
        }
    )

    data["GeneratedStoreID"] = data["StoreID"].map(
        {
            s: str(uuid.uuid4())
            for s in data["StoreID"].unique()
        }
    )

    data["SaleID"] = [
        str(uuid.uuid4())
        for _ in range(len(data))
    ]

    rates = {
        "usd": 0.385,
        "eur": 0.420,
        "omr": 1
    }

    data["Unit_Price_OMR"] = data.apply(
        lambda r:
        r["Unit_Price"] *
        rates.get(r["CurrencyType"], 1),
        axis=1
    )

    data["Total_Price"] = (
        data["Qty"] *
        data["Unit_Price_OMR"]
    )

    num_cols = [
        "Qty",
        "Unit_Price_OMR",
        "Total_Price"
    ]

    pipe = ColumnTransformer([
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="mean"
                    )
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            num_cols
        )
    ])

    data[num_cols] = pipe.fit_transform(
        data
    )

    return data


# =========================================================
# CREATE TABLES
# =========================================================

def create_tables():

    with engine.connect() as conn:

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Product (

            ProductID VARCHAR(255) PRIMARY KEY,

            ProductName VARCHAR(255)

        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Customer (

            CustomerPK VARCHAR(255) PRIMARY KEY,

            OriginalCustomerID VARCHAR(255)

        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Store (

            StorePK VARCHAR(255) PRIMARY KEY,

            OriginalStoreID VARCHAR(255)

        )
        """))

        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS Sale (

            SaleID VARCHAR(255) PRIMARY KEY,

            ProductID VARCHAR(255),

            CustomerPK VARCHAR(255),

            StorePK VARCHAR(255),

            Qty FLOAT,

            Unit_Price_OMR FLOAT,

            Total_Price FLOAT,

            SaleDate DATE,

            CurrencyType VARCHAR(50),

            Source_File VARCHAR(255)

        )
        """))

        conn.commit()

# =========================================================
# LOAD FUNCTION
# =========================================================

def load_data(data):

    product_df = data[
        ["ProductID", "ProductName"]
    ].drop_duplicates()

    customer_df = data[
        ["GeneratedCustomerID", "CustomerID"]
    ].drop_duplicates()

    customer_df.columns = [
        "CustomerPK",
        "OriginalCustomerID"
    ]

    store_df = data[
        ["GeneratedStoreID", "StoreID"]
    ].drop_duplicates()

    store_df.columns = [
        "StorePK",
        "OriginalStoreID"
    ]

    sale_df = data[[
        "SaleID",
        "ProductID",
        "GeneratedCustomerID",
        "GeneratedStoreID",
        "Qty",
        "Unit_Price_OMR",
        "Total_Price",
        "SaleDate",
        "CurrencyType",
        "Source_File"
    ]].copy()

    sale_df.columns = [
        "SaleID",
        "ProductID",
        "CustomerPK",
        "StorePK",
        "Qty",
        "Unit_Price_OMR",
        "Total_Price",
        "SaleDate",
        "CurrencyType",
        "Source_File"
    ]

    print("\nCHECKING FOR NEW RECORDS...")

    try:

        existing_count = pd.read_sql(
            "SELECT COUNT(*) AS cnt FROM Sale",
            engine
        ).iloc[0]["cnt"]

        print(
            f"Existing Records: {existing_count}"
        )

        sale_df = sale_df.iloc[
            int(existing_count):
        ]

        print(
            f"New Records: {len(sale_df)}"
        )

    except Exception as e:

        print(
            "First Run:",
            e
        )

    try:

        product_df.to_sql(
            "Product",
            engine,
            if_exists="append",
            index=False
        )

    except:
        pass

    try:

        customer_df.to_sql(
            "Customer",
            engine,
            if_exists="append",
            index=False
        )

    except:
        pass

    try:

        store_df.to_sql(
            "Store",
            engine,
            if_exists="append",
            index=False
        )

    except:
        pass

    if len(sale_df) > 0:

        sale_df.to_sql(
            "Sale",
            engine,
            if_exists="append",
            index=False
        )

        print(
            f"{len(sale_df)} new rows inserted."
        )

    else:

        print(
            "No new rows found."
        )

# =========================================================
# RUN ETL PIPELINE
# =========================================================

def run_etl_pipeline():

    print("\n=================================")
    print("STARTING ETL PIPELINE")
    print("=================================")

    # STEP 1 - EXTRACT
    data = extract_data(
        r"C:\Users\DELL\wajd-mq\sales"
    )

    # STEP 2 - TRANSFORM
    data = transform_data(data)

    # STEP 3 - CREATE TABLES
    create_tables()

    # STEP 4 - LOAD
    load_data(data)

    print("\n=================================")
    print("ETL PIPELINE COMPLETED")
    print("=================================")

# =========================================================
# AUTO RUN EVERY 3 MINUTES
# =========================================================

while True:

    try:

        print("\n")
        print("#################################")
        print("RUNNING ETL...")
        print("#################################")

        run_etl_pipeline()

        print("\nWAITING 1 MINUTES...")

    except Exception as e:

        print("\nERROR OCCURRED:")
        print(e)

    # 180 seconds = 3 minutes
    time.sleep(60)



#################################
RUNNING ETL...
#################################

STARTING ETL PIPELINE

EXTRACTING CSV FILES
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_1.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_2.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_3.csv

FILES MERGED SUCCESSFULLY

TRANSFORMING DATA

CHECKING FOR NEW RECORDS...
Existing Records: 269
New Records: 1
1 new rows inserted.

ETL PIPELINE COMPLETED

WAITING 1 MINUTES...


C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:340: UserWarning: The provided table name 'Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  product_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:352: UserWarning: The provided table name 'Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:364: UserWarning: The provided table name 'Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  store_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:376: UserWarning: The provided table name 'Sale' is not found exactly as such in the 



#################################
RUNNING ETL...
#################################

STARTING ETL PIPELINE

EXTRACTING CSV FILES
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_1.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_2.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_3.csv

FILES MERGED SUCCESSFULLY

TRANSFORMING DATA

CHECKING FOR NEW RECORDS...
Existing Records: 270
New Records: 0
No new rows found.

ETL PIPELINE COMPLETED

WAITING 1 MINUTES...


C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:340: UserWarning: The provided table name 'Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  product_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:352: UserWarning: The provided table name 'Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:364: UserWarning: The provided table name 'Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  store_df.to_sql(




#################################
RUNNING ETL...
#################################

STARTING ETL PIPELINE

EXTRACTING CSV FILES
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_1.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_2.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_3.csv

FILES MERGED SUCCESSFULLY

TRANSFORMING DATA

CHECKING FOR NEW RECORDS...
Existing Records: 270
New Records: 1
1 new rows inserted.

ETL PIPELINE COMPLETED

WAITING 1 MINUTES...


C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:340: UserWarning: The provided table name 'Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  product_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:352: UserWarning: The provided table name 'Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:364: UserWarning: The provided table name 'Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  store_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:376: UserWarning: The provided table name 'Sale' is not found exactly as such in the 



#################################
RUNNING ETL...
#################################

STARTING ETL PIPELINE

EXTRACTING CSV FILES
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_1.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_2.csv
Reading File: C:\Users\DELL\wajd-mq\sales\store_sales_3.csv

FILES MERGED SUCCESSFULLY

TRANSFORMING DATA

CHECKING FOR NEW RECORDS...
Existing Records: 271
New Records: 0
No new rows found.

ETL PIPELINE COMPLETED

WAITING 1 MINUTES...


C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:340: UserWarning: The provided table name 'Product' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  product_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:352: UserWarning: The provided table name 'Customer' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_df.to_sql(
C:\Users\DELL\AppData\Local\Temp\ipykernel_1196\2552462735.py:364: UserWarning: The provided table name 'Store' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  store_df.to_sql(


KeyboardInterrupt: 